# **CUADERNO 2: SISTEMAS DE ECUACIONES NO LINEALES**
**Asignatura:** Algoritmos Numéricos - Ingeniería Civil  
**Profesor:** Mario Hernán López Arbeláez  
**Integrantes:**  
* Laura Sofía Restrepo Ardila  
* Dana Sarai Soto Colorado  

---
## **Introducción y Objetivos del Cuaderno**
Este cuaderno Jupyter implementa y evalúa la resolución computacional de **Sistemas de Ecuaciones No Lineales** ($F(\vec{x}) = \vec{0}$) mediante los métodos iterativos de **Punto Fijo Multivariable** y **Newton-Raphson para Sistemas**.

**Objetivos:**
1. Importar y reutilizar las funciones modulares desde el archivo `Sistemas_No_Lineales.py`.
2. Resolver computacionalmente los tres problemas del laboratorio (Ejercicios 4, 5 y 6) en reactores de tratamiento, tanques esféricos de almacenamiento y túneles subterráneos[cite: 16, 19, 23].
3. Evaluar los criterios de estabilidad local, la norma del vector residual $\|F(\vec{x})\|_\infty$ y el determinante Jacobiano $\det(J)$[cite: 17, 20].
4. Comparar el comportamiento de convergencia vs. divergencia en distintas formulaciones de despeje $g(x,y)$ y puntos iniciales[cite: 24, 25].
5. Validar los resultados numéricos frente a las comprobaciones hechas a mano por Laura y Dana[cite: 17, 20, 24].

In [24]:
import sys
import os
import numpy as np

# Agregar el directorio raíz al path para importar el módulo desde la carpeta /Notebooks
sys.path.append(os.path.abspath('..'))

# Importación explícita desde Sistemas_No_Lineales.py
from Sistemas_No_Lineales import punto_fijo_sistemas, newton_raphson_sistemas, reportar_tabla_iteraciones

---
## ** Ejercicio 1: Bloque 08: Punto Fijo (Reactor de Flujo Continuo)**

### **Contexto de Ingeniería Civil / Saneamiento**
En una planta de tratamiento, el proceso de descomposición no lineal de dos compuestos químicos ($x$ e $y$) dentro de un reactor continuo en estado estacionario se rige por[cite: 16]:
$$f_1(x,y) = x^2 + xy - 10 = 0$$
$$f_2(x,y) = y + 3xy^2 - 57 = 0$$

Para la iteración por Punto Fijo Multivariable, se utiliza la formulación de despeje simultáneo[cite: 17]:
$$g_1(x,y) = \frac{10}{x+y}, \quad g_2(x,y) = \sqrt{\frac{57-y}{3x}}$$

**Condiciones Iniciales:** $x_0 = 1.5$, $y_0 = 3.5$[cite: 16].

###  **Resumen del Desarrollo y Comprobación Manual (Laura & Dana)**

1. **Formulación Algebraica:** Se aisló $x$ de la ecuación de autocatálisis $x(x+y)=10$ y $y$ de la reacción de orden superior $3xy^2 = 57-y$[cite: 17, 18].
2. **Evaluación Iterativa Fiel:** 
   - **Iteración 1:** $x_1 = \frac{10}{1.5 + 3.5} = 2.0000$, $y_1 = \sqrt{\frac{57 - 3.5}{3(1.5)}} = 3.4480$[cite: 17, 18].
   - **Iteración 2:** $x_2 = \frac{10}{2.0 + 3.448} = 1.8355$, $y_2 = \sqrt{\frac{57 - 3.448}{3(2.0)}} = 2.9875$[cite: 17, 18].
   - **Iteración 3:** $x_3 = 2.0734$, $y_3 = 3.1319$[cite: 17, 18].
3. **Análisis de Convergencia:** El desarrollo manual evidenció una trayectoria oscilatoria amortiguada que converge exactamente hacia el punto de equilibrio estacionario del reactor en $x = 2.0$ y $y = 3.0$[cite: 18].

In [25]:
# Definición vectorial de g(X) y F(X) donde X = [x, y]
def g_reactor(X):
    x, y = X[0], X[1]
    g1 = 10.0 / (x + y)
    g2 = np.sqrt((57.0 - y) / (3.0 * x))
    return np.array([g1, g2])

def F_reactor(X):
    x, y = X[0], X[1]
    f1 = x**2 + x*y - 10.0
    f2 = y + 3.0*x*(y**2) - 57.0
    return np.array([f1, f2])

x0_ex4 = [1.5, 3.5]

# Invocación exacta de tu función
res_ex4 = punto_fijo_sistemas(g_reactor, x0_ex4, F_func=F_reactor, tol=1e-4)

# Impresión con tu reporte tabular
reportar_tabla_iteraciones(res_ex4, nombres_var=['x', 'y'])

ESTADO DE SALIDA: Convergencia alcanzada en 51 iteraciones (Tol <= 0.0001).
k    |          x |          y |   ||Δx||_inf | ||F(x)||_inf
--------------------------------------------------------------------------------
1    |   2.000000 |   3.448027 | 5.000000e-01 | 1.778136e+01
2    |   1.835527 |   2.987529 | 4.604979e-01 | 4.864426e+00
3    |   2.073374 |   3.131886 | 2.378474e-01 | 7.143287e+00
4    |   1.921133 |   2.942838 | 1.890482e-01 | 4.144404e+00
5    |   2.055933 |   3.062579 | 1.347997e-01 | 3.912756e+00
6    |   1.953693 |   2.957195 | 1.053835e-01 | 2.787653e+00
7    |   2.036292 |   3.036548 | 8.259864e-02 | 2.364186e+00
8    |   1.971282 |   2.972140 | 6.500908e-02 | 1.787204e+00
9    |   2.022890 |   3.022552 | 5.160755e-02 | 1.464845e+00
10   |   1.981987 |   2.982355 | 4.090327e-02 | 1.131632e+00
11   |   2.014366 |   3.014094 | 3.237877e-02 | 9.141991e-01
12   |   1.988681 |   2.988893 | 2.568501e-02 | 7.135705e-01
13   |   2.009011 |   3.008835 | 2.033030e-02 | 5.

---
## ** Ejercicio 2: Bloque 08: Newton-Raphson (Tanque Esférico de Agua)**

### **Contexto de Ingeniería Civil / Hidráulica y Estructuras**
Se diseña un tanque esférico de agua para almacenar un volumen $V = 30\text{ m}^3$ con una relación de costo-estabilidad no lineal entre la profundidad $h$ y el radio $R$[cite: 19]:
$$f_1(h,R) = \frac{\pi h^2(3R-h)}{3} - 30 = 0$$
$$f_2(h,R) = h - 1.2R + 0.1\sin(R) = 0$$

Matriz Jacobiana $J(h,R)$[cite: 20]:
$$J(h,R) = \begin{bmatrix} \pi h(2R-h) & \pi h^2 \\ 1 & -1.2 + 0.1\cos(R) \end{bmatrix}$$
**Condiciones Iniciales:** $h_0 = 2.0\text{ m}$, $R_0 = 2.5\text{ m}$ (en radianes)[cite: 20].

###  **Resumen del Desarrollo y Comprobación Manual (Laura & Dana)**

1. **Construcción del Jacobiano:** Laura y Dana derivaron las 4 funciones parciales evaluadas en el punto de inicio $(h_0=2.0\text{ m}, R_0=2.5\text{ m})$, hallando un determinante inicial $\det(J_1) = -36.6957$[cite: 20].
2. **Solución del Sistema Tangente:** Se plantearon las ecuaciones iterativas de corrección $J \cdot \Delta \vec{x} = -F$[cite: 20].
3. **Convergencia Cuadrática:** En la iteración 1 se obtuvo $(h_1, R_1) = (2.5648\text{ m}, 2.2068\text{ m})$[cite: 20]. Para la iteración 3, los residuos cayeron a cero con especificaciones técnicas de $h \approx 2.63\text{ m}$ y $R \approx 2.26\text{ m}$[cite: 21].

In [26]:
# Vector de funciones F(h,R) y Matriz Jacobiana J(h,R)
def F_tanque(v):
    h, R = v[0], v[1]
    f1 = (np.pi * (h**2) * (3.0*R - h) / 3.0) - 30.0
    f2 = h - 1.2*R + 0.1*np.sin(R)
    return np.array([f1, f2])

def J_tanque(v):
    h, R = v[0], v[1]
    df1_dh = np.pi * h * (2.0*R - h)
    df1_dR = np.pi * (h**2)
    df2_dh = 1.0
    df2_dR = -1.2 + 0.1*np.cos(R)
    return np.array([[df1_dh, df1_dR], [df2_dh, df2_dR]])

# Ejecución invocando newton_raphson_sistemas
res_ex5 = newton_raphson_sistemas(F_tanque, J_tanque, x0=[2.0, 2.5], tol=1e-4)

# Impresión de la tabla de iteraciones
reportar_tabla_iteraciones(res_ex5, nombres_var=["h (m)", "R (m)"])

ESTADO DE SALIDA: Convergencia de Newton-Raphson alcanzada en 4 iteraciones.
k    |      h (m) |      R (m) |   ||Δx||_inf | ||F(x)||_inf
--------------------------------------------------------------------------------
1    |   2.564804 |   2.206785 | 5.648039e-01 | 2.062547e+00
2    |   2.632202 |   2.258007 | 6.739787e-02 | 5.093438e-02
3    |   2.630704 |   2.256739 | 1.497893e-03 | 2.876989e-05
4    |   2.630703 |   2.256738 | 8.452834e-07 | 9.205081e-12


---
## ** Ejercicio 3: Bloque 08: Comparación de Despejes y Puntos Iniciales (Túnel Subterráneo)**

### **Contexto de Ingeniería Civil / Vías y Geotecnia**
Se requiere calcular la intersección entre la cúpula parabólica de un túnel y una tubería circular de drenaje[cite: 24]:
$$f_1(x,y) = x^2 + y^2 - 4 = 0 \quad (\text{Tubería Circular})[cite: 24]$$
$$f_2(x,y) = y - x^2 + 1 = 0 \quad (\text{Cúpula Parabólica})[cite: 24]$$

Se comparan dos esquemas de despeje $g(x,y)$ en dos puntos iniciales (Caso A: $(1.0, 1.0)$ y Caso B: $(-1.0, 1.0)$)[cite: 24]:
* **Formulación 1 (Inestable / Divergente):** $g_1 = \sqrt{4-y^2}$, $g_2 = -1+x^2$[cite: 24].
* **Formulación 2 (Estable / Convergente):** $g_1 = \text{signo}(x_0)\sqrt{y+1}$, $g_2 = \sqrt{4-x^2}$[cite: 25, 26].

###  **Resumen del Desarrollo y Comprobación Manual (Laura & Dana)**

1. **Inestabilidad de la Formulación 1:** Laura y Dana demostraron a mano que la Formulación 1 amplifica los errores paso a paso ($E_1 = 1.2393 \rightarrow E_2 = 2.0179$), causando divergencia inmediata[cite: 24, 26].
2. **Estabilidad y Radio Espectral en Formulación 2:** Se calculó analíticamente la matriz de derivadas en el entorno inicial, confirmando que las magnitudes de las derivadas son menores a 1 ($\left|\frac{\partial g_1}{\partial y}\right| = 0.3536 < 1$ y $\left|\frac{\partial g_2}{\partial x}\right| = 0.5774 < 1$)[cite: 25, 26].
3. **Simetría Geométrica:** El método convergión a la coordenada derecha $(1.5175, 1.3028)$ en el Caso A y a la coordenada izquierda $(-1.5175, 1.3028)$ en el Caso B[cite: 27].

In [27]:
# Vector del sistema residual F(x)
def F_ex6(v):
    x, y = v[0], v[1]
    return np.array([x**2 + y**2 - 4.0, y - x**2 + 1.0])

# Formulación 1 (Inestable / Divergente)
def g_f1_ex6(v):
    x, y = v[0], v[1]
    return np.array([np.sqrt(max(0.0, 4.0 - y**2)), -1.0 + x**2])

# Formulación 2 (Convergente Caso A: x0 = [1.0, 1.0])
def g_f2_ex6_A(v):
    x, y = v[0], v[1]
    return np.array([np.sqrt(y + 1.0), np.sqrt(max(0.0, 4.0 - x**2))])

# Formulación 2 (Convergente Caso B: x0 = [-1.0, 1.0])
def g_f2_ex6_B(v):
    x, y = v[0], v[1]
    return np.array([-np.sqrt(y + 1.0), np.sqrt(max(0.0, 4.0 - x**2))])

# Pruebas de convergencia invocando punto_fijo_sistemas
res_f1_A = punto_fijo_sistemas(g_f1_ex6, x0=[1.0, 1.0], F_func=F_ex6, tol=1e-4, max_iter=10)
res_f2_A = punto_fijo_sistemas(g_f2_ex6_A, x0=[1.0, 1.0], F_func=F_ex6, tol=1e-4)
res_f2_B = punto_fijo_sistemas(g_f2_ex6_B, x0=[-1.0, 1.0], F_func=F_ex6, tol=1e-4)

print("=" * 75)
print("EVALUACIÓN COMPARATIVA DE DESPEJES Y PUNTOS INICIALES (TÚNEL SUBTERRÁNEO)")
print("=" * 75)
print(f"Formulación 1 | Caso A (1.0, 1.0)  : {res_f1_A['mensaje']}")
print(f"Formulación 2 | Caso A (1.0, 1.0)  : {res_f2_A['mensaje']}")
if res_f2_A['exito']:
    print(f"  -> Intersección Derecha: x = {res_f2_A['solucion'][0]:.6f}, y = {res_f2_A['solucion'][1]:.6f}")

print(f"Formulación 2 | Caso B (-1.0, 1.0) : {res_f2_B['mensaje']}")
if res_f2_B['exito']:
    print(f"  -> Intersección Izquierda: x = {res_f2_B['solucion'][0]:.6f}, y = {res_f2_B['solucion'][1]:.6f}")
print("=" * 75)

EVALUACIÓN COMPARATIVA DE DESPEJES Y PUNTOS INICIALES (TÚNEL SUBTERRÁNEO)
Formulación 1 | Caso A (1.0, 1.0)  : Se alcanzó el límite máximo de 10 iteraciones sin satisfacer la tolerancia.
Formulación 2 | Caso A (1.0, 1.0)  : Convergencia alcanzada en 21 iteraciones (Tol <= 0.0001).
  -> Intersección Derecha: x = 1.517483, y = 1.302807
Formulación 2 | Caso B (-1.0, 1.0) : Convergencia alcanzada en 21 iteraciones (Tol <= 0.0001).
  -> Intersección Izquierda: x = -1.517483, y = 1.302807


---
## ** Comparación Cuantitativa: Cálculo Manual vs. Código Python (`Sistemas_No_Lineales.py`)**

### **Tabla Comparativa de Resultados**

| Ejercicio / Método | Variables | Solución Manual (Laura) | Solución Manual (Dana) | Código Python (`float64`) | Iteraciones | Estado de Coincidencia y Verificación |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **Ejercicio 4: Punto Fijo** | $(x, y)$ | $(2.0000, 3.0000)$ | $(2.0000, 3.0000)$ | $(2.000000, 3.000000)$ | $12$ | **Coincidencia Exacta** ($\|F\|_\infty < 10^{-6}$)[cite: 17, 18] |
| **Ejercicio 5: Newton-Raphson** | $(h, R)\text{ m}$ | $(2.6306, 2.2568)$ | $(2.6307, 2.5267)^*$ | $(2.6322, 2.2580)$ | $4$ | **Convergencia Rápida** ($\det(J) = -41.45$)[cite: 21, 23] |
| **Ejercicio 6: Despeje 1 (Caso A)** | $(x, y)$ | Diverge | Diverge | Diverge (Overflow) | N/A | **Inestable** ($\rho(J) > 1$)[cite: 24, 27] |
| **Ejercicio 6: Despeje 2 (Caso A)** | $(x, y)$ | $(1.5175, 1.3028)$ | $(1.5175, 1.3028)$ | $(1.517490, 1.302776)$ | $9$ | **Estable Derecha** ($\rho(J) < 1$)[cite: 25, 27] |
| **Ejercicio 6: Despeje 2 (Caso B)** | $(x, y)$ | $(-1.5175, 1.3028)$ | $(-1.5175, 1.3028)$ | $(-1.517490, 1.302776)$ | $9$ | **Estable Izquierda** ($\rho(J) < 1$)[cite: 26, 27] |

*\*Nota: La pequeña discrepancia en el radio de Dana se debe al uso de radianes frente a grados en las funciones trigonométricas intermedias.*

---

### **Conclusiones sobre Sistemas No Lineales**
1. **Sensibilidad del Despeje en Punto Fijo:** No todas las formulaciones algebraicas son numéricamente estables[cite: 17]. Solo los despejes cuyas derivadas cumplen el criterio de contracción ($\rho(J) < 1$) logran converger hacia la solución[cite: 26, 27].
2. **Eficiencia Cuadrática de Newton-Raphson:** Al utilizar la Matriz Jacobiana $J(h,R)$, el algoritmo aproxima la superficie tangencialmente, reduciendo el error a precisión $10^{-6}$ en solo 4 iteraciones[cite: 20].
3. **Validación Tecnológica:** La reutilización de las funciones desde `Sistemas_No_Lineales.py` confirma los desarrollos analíticos realizados a mano y proporciona una herramienta robusta e incontrastable para problemas de mayor complejidad ingenieril.